# 03 — Fine-Tune Predictor

Fine-tunes the `Kronos` predictor model using the previously fine-tuned tokenizer.

**Prerequisite:** Run `02_train_tokenizer.ipynb` first so the tokenizer checkpoint exists.


In [ ]:
import sys, os
sys.path.insert(0, '..')  # repo root

SYMBOL      = 'bnbusdt'
TIMEFRAME   = '1m'
GCS_BUCKET  = 'epochquant-training'
GCS_PROJECT = None

DATA_PATH  = f'gs://{GCS_BUCKET}/processed/{SYMBOL}_{TIMEFRAME}.csv'
SAVE_PATH  = f'../output_models/{SYMBOL}_{TIMEFRAME}'

TOKENIZER_CKPT = os.path.join(SAVE_PATH, 'tokenizer_finetuned', 'checkpoints', 'best_model')
print('Tokenizer checkpoint:', TOKENIZER_CKPT)
print('Exists:', os.path.exists(TOKENIZER_CKPT))


In [ ]:
from training.config import Config

config = Config()
config.dataset_path           = DATA_PATH
config.gcs_project            = GCS_PROJECT
config.save_path              = SAVE_PATH
config.finetuned_tokenizer_path = TOKENIZER_CKPT
config.epochs                 = 3
config.batch_size             = 8
config.predictor_learning_rate = 1e-5

print('Config ready.')
print(f'  dataset_path           : {config.dataset_path}')
print(f'  finetuned_tokenizer_path: {config.finetuned_tokenizer_path}')


In [ ]:
# ── Load fine-tuned tokenizer and base predictor ─────────────────
import torch
from model.kronos import Kronos, KronosTokenizer

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = KronosTokenizer.from_pretrained(config.finetuned_tokenizer_path).to(device)
model     = Kronos.from_pretrained(config.pretrained_predictor_path).to(device)

tokenizer.eval()
for p in tokenizer.parameters():
    p.requires_grad = False

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Predictor trainable params: {params/1e6:.1f}M on {device}')


In [ ]:
# ── Run training ─────────────────────────────────────────────────
# For multi-GPU DDP, run from terminal:
#   torchrun --standalone --nproc_per_node=NUM_GPUS -m training.train_predictor
#
# For YAML-config based training (recommended for multi-asset):
#   python -m training.finetune_base_model --config configs/bnbusdt_1m.yaml

import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'training.train_predictor'],
    cwd='..', capture_output=False
)
print('Exit code:', result.returncode)


In [ ]:
# ── Verify and optionally upload checkpoint to GCS ───────────────
predictor_ckpt = os.path.join(SAVE_PATH, 'predictor_finetuned', 'checkpoints', 'best_model')
print('Predictor checkpoint:', predictor_ckpt)
print('Exists:', os.path.exists(predictor_ckpt))

# Upload to GCS (optional)
GCS_MODEL_PATH = f'gs://{GCS_BUCKET}/models/{SYMBOL}_{TIMEFRAME}/predictor/best_model'
print(f'\nTo upload to GCS:')
print(f'  gsutil -m cp -r {predictor_ckpt} {GCS_MODEL_PATH}')
